In [3]:
import pandas as pd
from pathlib import Path

series_df = pd.read_csv("../Data/prem_division_series_results.csv")


# CONVERT SERIES RESULTS TO 1 / -1 / 0

def get_series_result(row):
    if row["result"] == "TIE":
        return 0
    return 1 if row["result"] == row["team1"] else -1


series_df["result"] = series_df.apply(get_series_result, axis=1)

series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2",
        "ties",
        "games_played",
    ]
]


# GET EACH TEAM'S SERIES SCORE
# Series win = 1, series tie = 0.5, series loss = 0

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    (team1["result"] == 1).astype(float)
    + 0.5 * (team1["result"] == 0).astype(float)
)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    (team2["result"] == -1).astype(float)
    + 0.5 * (team2["result"] == 0).astype(float)
)

score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)


# LOOK UP EACH TEAM'S SCORE

score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# STRICT UPSETS
# A lower-scoring team beats a higher-scoring team.
# Tied series and equal-score teams do not count.

def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# WEAK UPSETS
# Same as strict, plus 0.5 for a tied series between
# teams with different total scores.

def is_weak_upset(row):
    if row["result"] == 0:
        s1 = score_lookup[
            (row["season"], row["division"], row["team1"])
        ]
        s2 = score_lookup[
            (row["season"], row["division"], row["team2"])
        ]
        return 0.5 if s1 != s2 else 0

    return row["upset"]


series_df["upset_weak"] = series_df.apply(
    is_weak_upset,
    axis=1
)

upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )[["upset", "upset_weak"]]
    .sum()
)


# SORT TEAMS BY DECREASING SCORE, AS IN THE MLB OUTPUT

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"],
    ascending=[True, True, False]
)


# COLLAPSE INTO ONE ROW PER SEASON / DIVISION

scoring_summary = (
    score_sequences
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )
    .agg(
        num_teams=("team", "size"),
        teams=("team", tuple),
        scoring_sequence=("wins", tuple)
    )
)


# GET THE NUMBER OF GAMES BETWEEN EACH PAIR

games_per_pair = (
    series_df
    .groupby(
        ["season", "division"],
        as_index=False
    )["games_played"]
    .first()
    .rename(columns={"games_played": "games_per_pair"})
)


# MERGE THE SUMMARIES

analysis_df = (
    scoring_summary
    .merge(
        games_per_pair,
        on=["season", "division"],
        how="left"
    )
    .merge(
        upset_counts,
        on=["season", "league", "division"],
        how="left"
    )
)

analysis_df = analysis_df.rename(
    columns={"upset": "upset_strict"}
)

analysis_df = analysis_df[
    [
        "season",
        "division",
        "num_teams",
        "games_per_pair",
        "teams",
        "scoring_sequence",
        "upset_strict",
        "upset_weak",
    ]
]


# SAVE THE OUTPUT

output_path = Path("../Analysis/premier_league_analysis.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(output_path, index=False)

print(f"Saved {len(analysis_df)} rows to {output_path}")

analysis_df

Saved 22 rows to ../Analysis/premier_league_analysis.csv


,season,division,num_teams,games_per_pair,teams,scoring_sequence,upset_strict,upset_weak
0,2001,Premier-League,20,2,"(ARS, LIVP, MANU, CHE, NEW, LEE, SOU, TOT, AVL...","(17.0, 14.5, 14.5, 13.5, 13.5, 13.0, 9.5, 9.5,...",25,49.0
1,2002,Premier-League,20,2,"(MANU, ARS, NEW, CHE, BLBN, LIVP, EVE, SOU, BI...","(15.5, 15.0, 13.0, 12.0, 11.5, 11.0, 10.5, 10....",21,53.0
2,2005,Premier-League,20,2,"(CHE, MANU, LIVP, TOT, ARS, BLBN, NEW, BOLN, W...","(16.5, 15.5, 14.5, 12.0, 11.5, 11.5, 11.0, 10....",24,48.5
3,2006,Premier-League,20,2,"(CHE, MANU, ARS, LIVP, EVE, TOT, POR, AVL, BLB...","(16.0, 16.0, 12.5, 12.5, 12.0, 11.5, 11.0, 10....",24,54.5
4,2007,Premier-League,20,2,"(CHE, MANU, ARS, LIVP, AVL, EVE, BLBN, MANC, P...","(16.5, 16.5, 15.5, 15.5, 12.0, 12.0, 10.5, 10....",15,34.5
5,2008,Premier-League,20,2,"(LIVP, MANU, CHE, ARS, AVL, EVE, FUL, MANC, WH...","(17.0, 16.5, 15.0, 13.5, 11.5, 11.0, 11.0, 9.0...",24,48.0
6,2009,Premier-League,20,2,"(CHE, MANU, MANC, ARS, TOT, AVL, LIVP, EVE, BL...","(15.5, 15.0, 14.5, 14.0, 13.5, 12.0, 12.0, 11....",23,48.5
7,2010,Premier-League,20,2,"(MANU, CHE, MANC, ARS, EVE, TOT, LIVP, FUL, AV...","(17.0, 13.0, 13.0, 12.5, 11.5, 11.0, 10.5, 10....",33,60.5
8,2011,Premier-League,20,2,"(MANC, MANU, ARS, TOT, NEW, CHE, EVE, FUL, WBA...","(16.0, 16.0, 13.5, 13.5, 12.5, 12.0, 10.5, 9.5...",26,52.5
9,2012,Premier-League,20,2,"(MANU, MANC, ARS, EVE, CHE, TOT, LIVP, WHU, NO...","(16.0, 15.0, 14.0, 14.0, 13.5, 13.0, 12.0, 9.0...",21,47.5
